# Projeto de Python para Finanças

In [ ]:
# instalação
%pip install yfinance pandas numpy plotly nbformat

### Parte 1: Obter cotações e construção de carteira

In [23]:
import yfinance as yf
import pandas as pd
from datetime import datetime, timedelta

acoes = ['ITUB4.SA', 'PETR4.SA', 'VALE3.SA', 'IVVB11.SA']
indices = ['^BVSP', '^GSPC', 'BRL=X', 'GC=F']

data_inicio = datetime.now() - timedelta(days=365)
data_inicio = data_inicio.strftime('%Y-%m-%d')
data_fim = datetime.now().strftime('%Y-%m-%d')
print(data_inicio, data_fim)

def pegar_cotacoes(lista_tickers, data_inicio, data_fim):
    df = yf.download(lista_tickers, data_inicio, data_fim, auto_adjust=False) # auto_adjust -> ajuste do preço de fechamento, decontando o valor do dividendo
    df = df['Adj Close']
    df = df.ffill() # preenche as linhas vazias com o valor anterior
    return df

lista_tickers = acoes + indices
df_cotacoes = pegar_cotacoes(lista_tickers, data_inicio, data_fim)
display(df_cotacoes)

[*********************100%***********************]  8 of 8 completed

2025-03-16 2026-03-16


Ticker,BRL=X,GC=F,ITUB4.SA,IVVB11.SA,PETR4.SA,VALE3.SA,^BVSP,^GSPC
Date,,,,,,,,
2025-03-17,5.740600,3000.000000,28.791889,361.100006,32.469765,52.420021,130834.0,5675.120117
2025-03-18,5.678069,3035.100098,28.946682,356.950012,32.496700,52.805603,131475.0,5614.660156
2025-03-19,5.672100,3035.899902,29.018381,358.899994,32.469765,52.713799,132508.0,5675.290039
2025-03-20,5.648100,3040.000000,28.749527,359.200012,32.541599,52.548550,131955.0,5662.890137
2025-03-21,5.673900,3018.199951,28.812260,362.649994,33.044449,52.741341,132345.0,5667.560059
...,...,...,...,...,...,...,...,...
2026-03-10,5.205300,5229.700195,43.799999,393.850006,42.930000,80.559998,183447.0,6781.479980
2026-03-11,5.162400,5167.399902,43.889999,393.299988,44.799999,79.849998,183969.0,6775.799805
2026-03-12,5.151400,5115.799805,42.689999,393.950012,45.000000,79.239998,179285.0,6672.620117


In [25]:
# carteira em termos de valores financeiros
dic_carteira = {
    'ITUB4.SA': 5000,
    'VALE3.SA': 3000,
    'PETR4.SA': 4000,
    'IVVB11.SA': 6000
}

df_carteira = pd.DataFrame()
total_investido = sum(dic_carteira.values())

print(total_investido)

for ativo in dic_carteira:
    preco_incial = df_cotacoes[ativo].iloc[0]
    qtde_acoes = dic_carteira[ativo] / preco_incial
    df_carteira[ativo] = df_cotacoes[ativo] * qtde_acoes

df_carteira['Total'] = df_carteira.sum(axis=1)

display(df_carteira)

18000


,ITUB4.SA,VALE3.SA,PETR4.SA,IVVB11.SA,Total
Date,,,,,
2025-03-17,5000.000000,3000.000000,4000.000000,6000.000000,18000.000000
2025-03-18,5026.881318,3022.066872,4003.318235,5931.044135,17983.310560
2025-03-19,5039.332592,3016.812897,4000.000000,5963.444827,18019.590316
2025-03-20,4992.643377,3007.355697,4008.849410,5968.429900,17977.278384
2025-03-21,5003.537539,3018.389133,4070.796219,6025.754435,18118.477326
...,...,...,...,...,...
2026-03-10,7606.308663,4610.452033,5288.612429,6544.170581,24049.543707
2026-03-11,7621.938091,4569.818756,5518.980459,6535.031534,24245.768841
2026-03-12,7413.545937,4534.908396,5543.618859,6545.832272,24037.905464


### Parte 2: Rentabilidade e Comparação com Benchmarks

In [19]:
df_cotacoes['SP500 (R$)'] = df_cotacoes['^GSPC'] * df_cotacoes['BRL=X']
df_cotacoes['OURO (R$)'] = df_cotacoes['GC=F'] * df_cotacoes['BRL=X']
df_cotacoes['Dólar'] = df_cotacoes['BRL=X']

df_cotacoes = df_cotacoes.drop(columns=['BRL=X', 'GC=F', '^GSPC'])
display(df_cotacoes)

Ticker,ITUB4.SA,IVVB11.SA,PETR4.SA,VALE3.SA,^BVSP,SP500 (R$),OURO (R$),Dólar
Date,,,,,,,,
2025-03-17,28.791889,361.100006,32.469765,52.420021,130834.0,32578.595164,17221.800327,5.740600
2025-03-18,28.946682,356.950012,32.496700,52.805603,131475.0,31880.428423,17233.508124,5.678069
2025-03-19,29.018381,358.899994,32.469765,52.713799,132508.0,32190.813012,17219.928040,5.672100
2025-03-20,28.749527,359.200012,32.541599,52.548550,131955.0,31984.569211,17170.223694,5.648100
2025-03-21,28.812260,362.649994,33.044449,52.741341,132345.0,32157.169739,17124.965088,5.673900
...,...,...,...,...,...,...,...,...
2026-03-10,43.799999,393.850006,42.930000,80.559998,183447.0,35299.636754,27222.157665,5.205300
2026-03-11,43.889999,393.299988,44.799999,79.849998,183969.0,34979.387345,26676.184061,5.162400
2026-03-12,42.689999,393.950012,45.000000,79.239998,179285.0,34373.335867,26353.531571,5.151400


In [20]:
def calcular_rentabilidade(df):
    retorno = df.iloc[-1] / df.iloc[0] - 1
    return retorno

display(calcular_rentabilidade(df_carteira))
display(calcular_rentabilidade(df_cotacoes))

ITUB4.SA     0.472637
VALE3.SA     0.493704
PETR4.SA     0.375741
IVVB11.SA    0.100249
Total        0.330487
dtype: float64

Ticker
ITUB4.SA      0.472637
IVVB11.SA     0.100249
PETR4.SA      0.375741
VALE3.SA      0.493704
^BVSP         0.357850
SP500 (R$)    0.067527
OURO (R$)     0.538446
Dólar        -0.086524
dtype: float64

In [26]:
df_comparacao = df_cotacoes.drop(columns=acoes)
df_comparacao['Carteira'] = df_carteira['Total']

display(df_comparacao)
print(calcular_rentabilidade(df_comparacao))

Ticker,BRL=X,GC=F,^BVSP,^GSPC,Carteira
Date,,,,,
2025-03-17,5.740600,3000.000000,130834.0,5675.120117,18000.000000
2025-03-18,5.678069,3035.100098,131475.0,5614.660156,17983.310560
2025-03-19,5.672100,3035.899902,132508.0,5675.290039,18019.590316
2025-03-20,5.648100,3040.000000,131955.0,5662.890137,17977.278384
2025-03-21,5.673900,3018.199951,132345.0,5667.560059,18118.477326
...,...,...,...,...,...
2026-03-10,5.205300,5229.700195,183447.0,6781.479980,24049.543707
2026-03-11,5.162400,5167.399902,183969.0,6775.799805,24245.768841
2026-03-12,5.151400,5115.799805,179285.0,6672.620117,24037.905464


Ticker
BRL=X      -0.086524
GC=F        0.684167
^BVSP       0.357850
^GSPC       0.168643
Carteira    0.330487
dtype: float64


### Parte 3: Análise de Risco

### Parte 4: Análise Técnica e Indicadores